### Flip horizontal, Rotate 15°, Brightness adjustment (+20%)

In [2]:
import os
import cv2
import shutil
import albumentations as A
from tqdm import tqdm

# Fungsi bantu baca dan simpan label YOLO
def read_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(float(parts[0]))  # konversi aman dari 1.0 → 1
            bbox = list(map(float, parts[1:]))
            boxes.append((cls, *bbox))
    return boxes

def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            line = f"{int(box[0])} {' '.join([f'{x:.6f}' for x in box[1:]])}\n"
            f.write(line)

# Augmentasi pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Konfigurasi path dan augmentasi
input_base = "dataset_75_15_10"
output_base = "dataset_75_15_10_augmented_2x"
target_size = (640, 640)
augment_times = 2  # Jumlah augmentasi per gambar
splits = ["train", "valid", "test"]

# Proses semua data
for split in splits:
    print(f"\n🔧 Processing split: {split}")

    input_img_dir = os.path.join(input_base, split, "images")
    input_lbl_dir = os.path.join(input_base, split, "labels")
    output_img_dir = os.path.join(output_base, split, "images")
    output_lbl_dir = os.path.join(output_base, split, "labels")

    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for fname in tqdm(image_files, desc=f"{split} images"):
        img_path = os.path.join(input_img_dir, fname)
        lbl_path = os.path.join(input_lbl_dir, os.path.splitext(fname)[0] + ".txt")

        if not os.path.exists(lbl_path):
            continue

        # Baca dan resize image asli
        image = cv2.imread(img_path)
        if image is None:
            print(f"⚠️ Gagal membaca: {img_path}")
            continue

        image = cv2.resize(image, target_size)
        boxes = read_yolo_label(lbl_path)
        if not boxes:
            continue

        # Simpan image dan label asli ke output (resize dulu)
        output_img_path = os.path.join(output_img_dir, fname)
        output_lbl_path = os.path.join(output_lbl_dir, os.path.splitext(fname)[0] + ".txt")
        cv2.imwrite(output_img_path, image)
        save_yolo_label(output_lbl_path, boxes)

        bboxes = [box[1:] for box in boxes]
        class_labels = [int(box[0]) for box in boxes]  # pastikan class int

        # Augmentasi
        for i in range(augment_times):
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = transformed["image"]
            aug_boxes = transformed["bboxes"]
            aug_labels = transformed["class_labels"]

            base_name = os.path.splitext(fname)[0]
            aug_img_name = f"{base_name}_aug{i}.jpg"
            aug_lbl_name = f"{base_name}_aug{i}.txt"

            cv2.imwrite(os.path.join(output_img_dir, aug_img_name), aug_img)
            save_yolo_label(
                os.path.join(output_lbl_dir, aug_lbl_name),
                [(int(cls), *bbox) for cls, bbox in zip(aug_labels, aug_boxes)]
            )

    print(f"✅ {split}: resize asli + {augment_times}x augmentasi selesai.")

# Menampilkan ringkasan jumlah file setelah augmentasi
print("\n📊 Ringkasan jumlah dataset setelah augmentasi:")
total_imgs, total_labels = 0, 0

for split in splits:
    img_dir = os.path.join(output_base, split, "images")
    lbl_dir = os.path.join(output_base, split, "labels")

    num_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    num_labels = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])

    total_imgs += num_imgs
    total_labels += num_labels

    print(f"📁 {split}: {num_imgs} gambar, {num_labels} label")

print(f"\n📦 Total: {total_imgs} gambar, {total_labels} label")




🔧 Processing split: train


train images:   0%|          | 0/1540 [00:00<?, ?it/s]

train images: 100%|██████████| 1540/1540 [03:05<00:00,  8.29it/s]


✅ train: resize asli + 2x augmentasi selesai.

🔧 Processing split: valid


valid images: 100%|██████████| 308/308 [00:36<00:00,  8.38it/s]


✅ valid: resize asli + 2x augmentasi selesai.

🔧 Processing split: test


test images: 100%|██████████| 205/205 [00:25<00:00,  8.03it/s]

✅ test: resize asli + 2x augmentasi selesai.

📊 Ringkasan jumlah dataset setelah augmentasi:
📁 train: 4620 gambar, 4620 label
📁 valid: 924 gambar, 924 label
📁 test: 615 gambar, 615 label

📦 Total: 6159 gambar, 6159 label


###  Simpan gambar original, +1 gambar hasil flip horizontal, +1 gambar hasil flip vertical

In [2]:
import os
import cv2
import shutil
import albumentations as A
from tqdm import tqdm

# Fungsi bantu baca dan simpan label YOLO
def read_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(float(parts[0]))  # konversi aman jika ada float
            bbox = list(map(float, parts[1:]))
            boxes.append((cls, *bbox))
    return boxes

def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            line = f"{int(box[0])} {' '.join([f'{x:.6f}' for x in box[1:]])}\n"
            f.write(line)

# Konfigurasi path dan transformasi
input_base = "dataset_clean"
output_base = "dataset_clean_flip"
target_size = (640, 640)
splits = ["train", "valid", "test"]

# Definisi transformasi
flip_h_transform = A.Compose([
    A.HorizontalFlip(p=1.0)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

flip_v_transform = A.Compose([
    A.VerticalFlip(p=1.0)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Untuk menghitung total
summary = {}

for split in splits:
    print(f"\n🔧 Processing split: {split}")

    input_img_dir = os.path.join(input_base, split, "images")
    input_lbl_dir = os.path.join(input_base, split, "labels")
    output_img_dir = os.path.join(output_base, split, "images")
    output_lbl_dir = os.path.join(output_base, split, "labels")

    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    original_count = 0

    for fname in tqdm(image_files, desc=f"{split} images"):
        img_path = os.path.join(input_img_dir, fname)
        lbl_path = os.path.join(input_lbl_dir, os.path.splitext(fname)[0] + ".txt")

        if not os.path.exists(lbl_path):
            continue

        image = cv2.imread(img_path)
        if image is None:
            print(f"⚠️  Gagal membaca gambar: {img_path}")
            continue

        image = cv2.resize(image, target_size)
        boxes = read_yolo_label(lbl_path)
        if not boxes:
            continue

        bboxes = [box[1:] for box in boxes]
        class_labels = [int(box[0]) for box in boxes]  # pastikan int

        # Simpan versi resized dari gambar original
        cv2.imwrite(os.path.join(output_img_dir, fname), image)
        save_yolo_label(os.path.join(output_lbl_dir, os.path.splitext(fname)[0] + ".txt"), boxes)
        original_count += 1

        # Flip Horizontal
        flipped_h = flip_h_transform(image=image, bboxes=bboxes, class_labels=class_labels)
        fliph_labels = [(int(cls), *bbox) for cls, bbox in zip(flipped_h["class_labels"], flipped_h["bboxes"])]
        cv2.imwrite(os.path.join(output_img_dir, f"{os.path.splitext(fname)[0]}_fliph.jpg"), flipped_h["image"])
        save_yolo_label(os.path.join(output_lbl_dir, f"{os.path.splitext(fname)[0]}_fliph.txt"), fliph_labels)

        # Flip Vertical
        flipped_v = flip_v_transform(image=image, bboxes=bboxes, class_labels=class_labels)
        flipv_labels = [(int(cls), *bbox) for cls, bbox in zip(flipped_v["class_labels"], flipped_v["bboxes"])]
        cv2.imwrite(os.path.join(output_img_dir, f"{os.path.splitext(fname)[0]}_flipv.jpg"), flipped_v["image"])
        save_yolo_label(os.path.join(output_lbl_dir, f"{os.path.splitext(fname)[0]}_flipv.txt"), flipv_labels)

    total_output = original_count * 3
    summary[split] = (original_count, total_output)
    print(f"✅ {split}: {original_count} gambar asli → {total_output} total (termasuk flip)")

# Cetak ringkasan akhir
print("\n📊 Ringkasan Dataset:")
for split, (ori, total) in summary.items():
    print(f"🗂 {split.capitalize()}: {ori} gambar asli → {total} total gambar")



🔧 Processing split: train


train images: 100%|██████████| 1310/1310 [01:53<00:00, 11.55it/s]


✅ train: 1310 gambar asli → 3930 total (termasuk flip)

🔧 Processing split: valid


valid images: 100%|██████████| 363/363 [00:32<00:00, 11.26it/s]


✅ valid: 363 gambar asli → 1089 total (termasuk flip)

🔧 Processing split: test


test images:  78%|███████▊  | 148/190 [00:14<00:03, 11.97it/s]

⚠️  Gagal membaca gambar: dataset_clean\test\images\fa5a0e63-train814_-_副本.jpg


test images: 100%|██████████| 190/190 [00:18<00:00, 10.03it/s]

✅ test: 189 gambar asli → 567 total (termasuk flip)

📊 Ringkasan Dataset:
🗂 Train: 1310 gambar asli → 3930 total gambar
🗂 Valid: 363 gambar asli → 1089 total gambar
🗂 Test: 189 gambar asli → 567 total gambar


In [6]:
import os

def count_images(folder_path, extensions=(".jpg", ".jpeg", ".png")):
    count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(extensions):
                count += 1
    return count

def estimate_yolo_ram_usage(num_images, image_size, dtype_bytes=4, channels=3, margin=0.5):
    bytes_per_image = image_size * image_size * channels * dtype_bytes
    total_bytes = num_images * bytes_per_image
    total_gb = total_bytes / (1024 ** 3)
    total_with_margin = total_gb * (1 + margin)
    return total_gb, total_with_margin

def recommend_cache_setting(required_gb, available_gb):
    if required_gb <= available_gb:
        return "✅ Aman menggunakan cache=True"
    else:
        return "⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar"

if __name__ == "__main__":
    # ==== 🔧 GANTI SESUAI KEBUTUHAN ====
    dataset_path = r"F:\Kuliah\Skripsi\Code\river-trash-monitoring\dataset_75_15_10_augmented_3x\train\images"
    available_ram_gb = 10  # RAM kosong di sistem kamu
    dtype_bytes = 4        # float32
    margin = 0.5           # 50% safety margin
    channels = 3           # RGB
    resolutions = [416, 512, 640, 768]
    # ==================================

    num_images = count_images(dataset_path)
    print(f"\n📁 Jumlah gambar ditemukan: {num_images} di folder {dataset_path}\n")

    print("📊 Estimasi kebutuhan RAM (per resolusi):\n")
    for res in resolutions:
        base_gb, with_margin_gb = estimate_yolo_ram_usage(
            num_images, res, dtype_bytes, channels, margin
        )
        recommendation = recommend_cache_setting(with_margin_gb, available_ram_gb)
        print(f"- Resolusi {res}x{res}:")
        print(f"  ➤ Tanpa margin: {base_gb:.2f} GB")
        print(f"  ➤ Dengan margin: {with_margin_gb:.2f} GB")
        print(f"  ➤ Rekomendasi: {recommendation}\n")



📁 Jumlah gambar ditemukan: 6160 di folder F:\Kuliah\Skripsi\Code\river-trash-monitoring\dataset_75_15_10_augmented_3x\train\images

📊 Estimasi kebutuhan RAM (per resolusi):

- Resolusi 416x416:
  ➤ Tanpa margin: 11.91 GB
  ➤ Dengan margin: 17.87 GB
  ➤ Rekomendasi: ⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar

- Resolusi 512x512:
  ➤ Tanpa margin: 18.05 GB
  ➤ Dengan margin: 27.07 GB
  ➤ Rekomendasi: ⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar

- Resolusi 640x640:
  ➤ Tanpa margin: 28.20 GB
  ➤ Dengan margin: 42.30 GB
  ➤ Rekomendasi: ⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar

- Resolusi 768x768:
  ➤ Tanpa margin: 40.61 GB
  ➤ Dengan margin: 60.91 GB
  ➤ Rekomendasi: ⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar

